# 🚲 Step 0 — Train ML Model (Run This ONCE Before Anything Else)
This notebook trains a Random Forest on the Bike Sharing dataset and saves the model.
Run all cells top to bottom. At the end, `bike_model.joblib` will be saved to your Drive.

In [ ]:
# Install dependencies
!pip install scikit-learn joblib pandas --quiet

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder
import numpy as np

# ─── CHANGE THIS PATH if your hour.csv is in a different folder ───
DATASET_PATH = '/content/drive/MyDrive/kafka-assignment/hour.csv'
MODEL_PATH   = '/content/drive/MyDrive/kafka-assignment/bike_model.joblib'
# ──────────────────────────────────────────────────────────────────

df = pd.read_csv(DATASET_PATH)
print(f'Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns')
print(df.head(3))

In [ ]:
# Feature engineering
FEATURES = ['season', 'yr', 'mnth', 'hr', 'holiday', 'weekday',
            'workingday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed']
TARGET = 'cnt'

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {len(X_train)} rows | Test: {len(X_test)} rows')

In [ ]:
# Train Random Forest
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
print('Model trained!')

In [ ]:
# Evaluate
y_pred = model.predict(X_test)
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)
# Approximate F1 via bucketing (low/med/high)
from sklearn.metrics import f1_score
def bucket(vals):
    return pd.cut(vals, bins=[0, 50, 150, 1000], labels=[0,1,2]).astype(int)
f1 = f1_score(bucket(y_test), bucket(pd.Series(y_pred)), average='weighted')

print('='*40)
print(f'  MAE  : {mae:.2f} bikes')
print(f'  R²   : {r2:.4f}')
print(f'  F1   : {f1:.4f}  (bucketed low/med/high)')
print('='*40)
print('COPY THESE NUMBERS INTO YOUR README.md!')

In [ ]:
# Save model + feature list together
artifact = {'model': model, 'features': FEATURES}
joblib.dump(artifact, MODEL_PATH)
print(f'Model saved to: {MODEL_PATH}')
print('Now download bike_model.joblib from Drive and re-upload it to your kafka-assignment folder.')